# NTX Q1 Stage 24 — Unified Q1 Evidence Audit, Claim Gate, and Final ZIP

Runs last. It audits every new stage, prevents unsupported manuscript claims, generates a paper-ready evidence table, and creates one final downloadable ZIP containing all completed outputs.

**Evidence rule:** failed/incomplete third-party infrastructure is recorded as such; no missing score is converted into a model failure or a success.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, time, hashlib, zipfile, random
BUNDLE = Path(os.environ.get("NTX_NOTEBOOK_BUNDLE", ".")).resolve()
if str(BUNDLE) not in sys.path:
    sys.path.insert(0, str(BUNDLE))
from ntx_q1_shared import *
ROOT = ensure_dir(DEFAULT_ROOT)
RESULTS_ROOT = ensure_dir(DEFAULT_RESULTS)
SEED = int(os.environ.get("NTX_SEED", "42"))
random.seed(SEED)
print("ROOT:", ROOT)
print("RESULTS:", RESULTS_ROOT)
print("Frozen repo commit:", REPO_COMMIT)

In [ ]:
import pandas as pd, numpy as np
STAGE="24_unified_evidence"
OUT=ensure_dir(RESULTS_ROOT/STAGE)
stage_dirs=[p for p in RESULTS_ROOT.iterdir() if p.is_dir() and p.name[:2].isdigit() and p.name!="24_unified_evidence"]
status_rows=[]
for d in sorted(stage_dirs):
    sp=d/"STATUS.json"
    if sp.exists():
        x=json.loads(sp.read_text())
        status_rows.append({"stage_dir":d.name,**x})
    else:
        status_rows.append({"stage_dir":d.name,"status":"NO_STATUS"})
status=pd.DataFrame(status_rows)
status.to_csv(OUT/"stage_status_matrix.csv",index=False)
display(status[["stage_dir","status"]])

In [ ]:
# Claim gate: supported only by observed artifacts, never by planned notebooks.
claims=[]

def has(stage,file):
    return (RESULTS_ROOT/stage/file).exists()

# Human-written
human_ok=False
if has("17_human_e2e","human_e2e_metrics.json") and has("17_human_e2e","agreement.json"):
    hm=json.loads((RESULTS_ROOT/"17_human_e2e/human_e2e_metrics.json").read_text())
    ha=json.loads((RESULTS_ROOT/"17_human_e2e/agreement.json").read_text())
    human_ok=hm.get("n",0)>=300 and ha.get("n_annotators",0)>=2
claims.append({"claim":"Human-written multilingual external authorization validation","supported":human_ok})

# AgentDojo
dojo_ok=False
if has("18_agentdojo_guarded","agentdojo_guarded_summary.json"):
    ds=json.loads((RESULTS_ROOT/"18_agentdojo_guarded/agentdojo_guarded_summary.json").read_text())
    dojo_ok=ds.get("n_pairs",0)>=50
claims.append({"claim":"Paired guarded-vs-unguarded AgentDojo authorization evaluation","supported":dojo_ok})

# Baselines
base_ok=has("19_baselines","paired_comparisons.csv")
claims.append({"claim":"Direct full-vs-single-best/policy baseline comparison","supported":base_ok})

# Adaptive redteam
red_ok=False
if has("20_adaptive_redteam","adaptive_redteam_metrics.json"):
    rm=json.loads((RESULTS_ROOT/"20_adaptive_redteam/adaptive_redteam_metrics.json").read_text())
    red_ok=rm.get("n_attacks",0)>=5000 and rm.get("clean_controls",0)>=1000 and rm.get("suite_frozen_before_eval",False)
claims.append({"claim":"Post-v3 unseen/frozen adaptive semantic red-team result","supported":red_ok})

# Broker
broker_ok=False
if has("21_broker_e2e","broker_e2e_summary.json"):
    bm=json.loads((RESULTS_ROOT/"21_broker_e2e/broker_e2e_summary.json").read_text())
    broker_ok=bm.get("mandatory_mediation_tested",False) and bm.get("concurrency_cases",0)>=500 and bm.get("unsafe_commits",1)==0
claims.append({"claim":"Mandatory transaction-bound execution under concurrency","supported":broker_ok})

# Formal
formal_ok=False
if has("23_formal_property_tests","monotonicity_summary.json"):
    fm=json.loads((RESULTS_ROOT/"23_formal_property_tests/monotonicity_summary.json").read_text())
    formal_ok=fm.get("property_holds",False) and fm.get("trials",0)>=10000
claims.append({"claim":"Empirical property test of monotonic authority contraction","supported":formal_ok})

claim_df=pd.DataFrame(claims)
claim_df.to_csv(OUT/"Q1_CLAIM_EVIDENCE_GATE.csv",index=False)
display(claim_df)

In [ ]:
# Generate a compact manuscript-integration report.
lines=["# NiyamTrace-X Q1 Upgrade Evidence Audit",""]
for _,r in claim_df.iterrows():
    lines.append(f"- {'SUPPORTED' if r.supported else 'NOT YET SUPPORTED'} — {r['claim']}")
lines += [
    "",
    "## Rule",
    "Only claims marked SUPPORTED above should be promoted into the manuscript as completed evidence.",
    "Planned notebooks, infrastructure failures, missing labels, or missing predictions are not evidence."
]
(OUT/"MANUSCRIPT_EVIDENCE_UPDATE.md").write_text("\n".join(lines),encoding="utf-8")
record_status(OUT,STAGE,"PASS",supported_claims=int(claim_df.supported.sum()),total_claims=len(claim_df))

In [ ]:
# Package all stage result directories plus this audit.
artifact_manifest(RESULTS_ROOT)
final_zip=RESULTS_ROOT.parent/"NTX_Q1_UPGRADE_ALL_RESULTS.zip"
zip_tree(RESULTS_ROOT,final_zip)
print("FINAL RESULTS ZIP:",final_zip)
print("SHA256:",sha256_file(final_zip))
maybe_colab_download(final_zip)